# RQ-VAE 3000-epoch 训练 — Colab

**运行前**：Runtime → Change runtime type → **GPU**（A100 优先，T4 也行）

**需要手动上传**：
- `embedding/item_embeddings_raw.npy`（本地 ~74 MB，gitignore）

**产出**：
- `checkpoints/rqvae_best.pt`
- `embedding/semantic_ids_rqvae.npy`

**预计耗时**：A100 ~1-2 h

---

## 运行顺序
1. Cell 1-4：环境 + clone + 上传 embedding
2. Cell 5：训练 RQ-VAE（主耗时）
3. Cell 6：生成 semantic IDs
4. Cell 7：下载 ckpt + sids 回本地

In [ ]:
# Cell 1：确认 GPU 可用
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  未检测到 GPU，请先到 Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2：安装依赖（RQ-VAE 只需要 torch + numpy + scikit-learn）
!pip install scikit-learn -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：克隆仓库
!git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
%cd Generative-Sequential-Recommendation
!pwd && ls

In [ ]:
# Cell 4：上传 item_embeddings_raw.npy（~74 MB）
import os
from google.colab import files

os.makedirs('embedding', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('请上传 item_embeddings_raw.npy（本地项目 embedding/ 目录里）')
uploaded = files.upload()
for fname in uploaded:
    dest = f'embedding/{fname}'
    os.rename(fname, dest)
    print(f'✅ 已移动到 {dest}  ({os.path.getsize(dest)/1e6:.1f} MB)')

import numpy as np
raw = np.load('embedding/item_embeddings_raw.npy', allow_pickle=True).item()
vals = list(raw.values())
print(f'\n✅ 加载成功：{len(raw)} items × {len(vals[0])}d')

In [ ]:
# Cell 5：训练 RQ-VAE（3000 epoch + 50 warmup，A100 约 1-2h）
# 配置见 embedding/rqvae.py：K_LEVELS=[256,256,256], batch=1024, lr=1e-3
# 输出：checkpoints/rqvae_best.pt
!python embedding/rqvae.py

In [ ]:
# Cell 6：生成 semantic IDs（加 c4 做 collision resolution）
# 输出：embedding/semantic_ids_rqvae.npy
!python embedding/generate_rqvae_ids.py

---
## 下载结果

In [ ]:
# Cell 7：下载 ckpt + sids 回本地
import os
from google.colab import files

for fpath in [
    'checkpoints/rqvae_best.pt',
    'embedding/semantic_ids_rqvae.npy',
]:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'✅ 下载：{fpath}')
    else:
        print(f'⏭️  跳过：{fpath}  (不存在)')